# 5. Real accelerometer–gyroscope validation

**Question:** Does a temporal-neighborhood intervention influence independently learned sensor representations beyond a small synthetic system?

Use the raw [UCI HAPT recordings](https://archive.ics.uci.edu/dataset/341/smartphone+based+recognition+of+human+activities+and+postural+transitions): 30 participants, three-axis acceleration and angular velocity at 50 Hz. The raw archive, not the precomputed 561-feature table, is required. Attribution: Reyes-Ortiz et al. (2015), DOI 10.24432/C54G7M, CC BY 4.0.

The accelerometer and gyroscope encoders train independently. Same-time pairing is used only for frozen calibration/evaluation. This validates an **empirical positive-pair kernel**, not matched entropy or spectrum of unknown natural dynamics.

**Default safety:** download, training, disk saving, and browser downloads are disabled. The loader can read an archive entirely in RAM. Editor autosave of displayed outputs is separate; disable it or clear outputs if you do not want them stored in the notebook.

In [ ]:
from pathlib import Path
import sys
import subprocess
from zipfile import ZipFile

REPO_URL = 'https://github.com/AlexM3020/The-Role-of-Dynamical-Structure-in-Cross-Modal-Representation-Learning.git'
REPO_REF = 'main'  # For a frozen run, choose a commit before starting a fresh runtime.
SOURCE_ARCHIVE = None  # Optional uploaded source ZIP; not needed in Colab with GitHub access.
INSTALL_DEPENDENCIES = False  # Enable only if the runtime lacks compatible dependencies.
cloud = next((p for p in (Path('/content'), Path('/kaggle/working')) if p.exists()), None)
if SOURCE_ARCHIVE is not None:
    if cloud is None:
        raise RuntimeError('Use the local checkout here; source extraction is for cloud runtimes.')
    destination = cloud / 'dynalign_uploaded_source'
    with ZipFile(SOURCE_ARCHIVE) as archive:
        for name in archive.namelist():
            if not (destination / name).resolve().is_relative_to(destination.resolve()):
                raise ValueError('Unsafe archive member')
        archive.extractall(destination)
    candidates = [destination]
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / 'src/dynalign/revised.py').exists()), None)
if ROOT is None and SOURCE_ARCHIVE is None and cloud is not None:
    destination = cloud / 'dynalign_github'
    if not destination.exists():
        subprocess.check_call(['git', 'clone', '--quiet', REPO_URL, str(destination)])
        subprocess.check_call(['git', '-C', str(destination), 'fetch', '--quiet', 'origin', REPO_REF])
        subprocess.check_call(['git', '-C', str(destination), 'checkout', '--quiet', '--detach', 'FETCH_HEAD'])
    ROOT = destination
if ROOT is None or not (ROOT / 'src/dynalign/revised.py').exists():
    raise FileNotFoundError('Open the current checkout locally, or use a fresh Colab/Kaggle runtime with Internet enabled.')
# Existing checkouts are deliberately reused, not reset or silently updated.
# Restart in a fresh runtime to switch REPO_REF or load a newer GitHub revision.
if 'dynalign' in sys.modules and Path(sys.modules['dynalign'].__file__).resolve().parents[2] != ROOT.resolve():
    raise RuntimeError('Another source version is already imported; restart the runtime before switching.')
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(ROOT)])
sys.path.insert(0, str(ROOT / 'src'))
revision = subprocess.run(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], capture_output=True, text=True)
print('Source revision:', revision.stdout.strip() if revision.returncode == 0 else 'uploaded source bundle')
print('Source setup only: result saving remains disabled by default.')

from dataclasses import replace
import pandas as pd
import matplotlib.pyplot as plt
from dynalign.hapt import WindowSettings, RealSpec, load_hapt, split_and_standardize, run_real
from dynalign.revised import FitSettings, EvalSettings, paired_contrasts, bundle_zip
print('Real-sensor engine loaded; no dataset has been downloaded.')

## Experiments to try

1. **Data audit:** enable loading only. Inspect participant/recording counts, paired channels, label coverage and non-overlapping windows before any training.
2. **Small execution check:** one seed, 256–512 temporal pairs, 40–80 updates, small CNN. Limited early-recording windows are only for debugging; activity probes may be undefined.
3. **Primary temporal comparison:** full recordings, local versus within-recording scrambled pairs, the same objective and training budget; approximately five seeds.
4. **Collapse cross-check:** compare the original regime and the candidate frozen from the synthetic objective investigation. Inspect rank together with alignment; do not select a real-data winner from test scores.
5. **Architecture boundary:** optional order-invariant pointwise encoder or linear baseline. Keep the condition comparison within each architecture; parameter counts differ across architectures.
6. **Shortcut diagnostic:** report CKA per held-out participant and within participant/activity groups, alongside frozen activity probes. Global alignment alone can reflect coarse activity or identity.

The default partition below is an explicit new participant split, **not the original official split**. Freeze it before confirmation. Normalization uses training participants only. Calibration participants fit the orthogonal map; test participants never fit it. Labels are used only for post-training probes/grouped evaluation; they do not choose temporal neighbors or enter encoder losses.

In [ ]:
# All primary settings are editable in this cell.
LOAD_DATA = False
ALLOW_DOWNLOAD = False  # True permits an in-memory UCI download only when LOAD_DATA=True.
HAPT_ARCHIVE_PATH = None  # Alternative: a previously uploaded raw HAPT archive.
MAX_ARCHIVE_BYTES = 600_000_000
RUN_TRAINING = False
SAVE_RESULTS, OUTPUT_DIR = False, None
DEVICE, CPU_THREADS = 'auto', 2
TRAIN_SUBJECTS = list(range(1, 19))
CALIBRATION_SUBJECTS = list(range(19, 25))
TEST_SUBJECTS = list(range(25, 31))
WINDOW = WindowSettings(length=128, stride=128, label_purity=0.8, max_windows_per_record=64)
# For meaningful activity evaluation use max_windows_per_record=None: do not keep only recording prefixes.
SEEDS = [0]  # Expanded: [0, 1, 2, 3, 4]
MASTER_SEED = 20261108
CONDITIONS = ['local', 'scrambled']
ARCHITECTURES = ['temporal_cnn']  # Optional 'order_invariant' or 'linear' in a separate declared analysis.
WIDTH, EMBEDDING_DIM, N_PAIRS = 16, 8, 512
ACTIVITY_LABELS = (1, 2, 3, 4, 5, 6)
MINIMUM_GROUP_SIZE = 32
FIT = FitSettings(steps=80, checkpoints=(40, 80), batch_size=128, lr=1e-3,
                  weight_decay=1e-4, invariance_weight=25, variance_weight=25,
                  covariance_weight=1, identity_weight=25, variance_target=1,
                  epsilon=1e-4, clip_grad=5, log_every=40, checkpoint_every=100)
EVAL = EvalSettings(calibration_repeats=8, test_repeats=16, ridge=1e-3,
                    collapse_variance=1e-6, collapse_rank_fraction=0.5, batch_size=512)
# Synthetic repeat-count fields are not used for real data; actual windows define the splits.
REGIMES = {
    'baseline': dict(objective='vicreg', invariance_weight=25, variance_weight=25, covariance_weight=1),
    'identity_candidate': dict(objective='cov_identity', invariance_weight=25, identity_weight=25),
}
ACTIVE_REGIMES = ['baseline', 'identity_candidate']  # Candidate is not an empirically selected winner.
if SAVE_RESULTS and OUTPUT_DIR is None:
    raise ValueError('Choose an explicit persistence directory before saving.')
specs = [RealSpec(repeat=seed, master_seed=MASTER_SEED, condition=condition, regime=regime,
                  architecture=architecture, width=WIDTH, dim=EMBEDDING_DIM, n_pairs=N_PAIRS,
                  valid_activity_labels=ACTIVITY_LABELS, minimum_group_size=MINIMUM_GROUP_SIZE,
                  fit=replace(FIT, **REGIMES[regime]), evaluation=EVAL)
         for seed in SEEDS for architecture in ARCHITECTURES for regime in ACTIVE_REGIMES for condition in CONDITIONS]
print(f'{2 * len(specs)} encoder fits in the requested plan. Load: {LOAD_DATA}; train: {RUN_TRAINING}; save: {SAVE_RESULTS}')

## Load, pair, and split raw observations

Non-overlapping windows use identical sample bounds for acceleration and gyroscope. Their participant and recording identifiers are retained. The loader checks finite values and paired shapes. Ambiguous/mixed-label windows remain in unsupervised training but receive evaluation label 0.

Within each recording, local positive pairs follow a symmetric reflecting neighbor graph. Scrambling conjugates that graph by a permutation while leaving the windows unchanged. It preserves this empirical graph's stationary measure and spectrum. It does not preserve all real temporal statistics, remove all nuisance variables, or infer the true physical transition law. Each modality uses independent pair draws; no pair crosses recordings.

In [ ]:
data = train = calibration = test = normalization = None
if LOAD_DATA:
    data = load_hapt(archive_path=HAPT_ARCHIVE_PATH, allow_download=ALLOW_DOWNLOAD,
                     window=WINDOW, max_bytes=MAX_ARCHIVE_BYTES)
    train, calibration, test, normalization = split_and_standardize(
        data, TRAIN_SUBJECTS, CALIBRATION_SUBJECTS, TEST_SUBJECTS)
    counts = pd.DataFrame({'subject': data.subject, 'recording': data.recording, 'activity': data.label})
    display(counts.groupby(['subject', 'activity']).size().unstack(fill_value=0))
    print('Split windows:', len(train.subject), len(calibration.subject), len(test.subject))
    print('Dataset provenance:', data.provenance)
else:
    print('Data loading disabled. Enable LOAD_DATA plus ALLOW_DOWNLOAD, or provide HAPT_ARCHIVE_PATH.')

## Train independent sensor encoders

The frozen activity readout is fitted on labeled **training-participant** embeddings after representation learning; no activity label trains the encoders. The orthogonal map is fitted on calibration participants. Same-time test pairing evaluates geometry only.

**Loss settings in `REGIMES` override corresponding `FIT` defaults.** Edit the regime dictionary for loss weights; edit `FIT` for optimizer/update choices. Complete merged specifications accompany the results.

Step-zero random-initialization measurements are retained. No test metric selects objectives, update counts, or checkpoints. With persistence disabled, all models and metrics remain in RAM; no real results are replaced with a synthetic fallback if data are missing.

In [ ]:
bundle = None
if RUN_TRAINING:
    if train is None:
        raise RuntimeError('Load and inspect real HAPT data first; no synthetic substitution is made.')
    bundle = run_real(specs, train, calibration, test, normalization=normalization,
                      device=DEVICE, threads=CPU_THREADS, output_dir=OUTPUT_DIR if SAVE_RESULTS else None)
    display(bundle.table)
else:
    print('Real-data training disabled. Enable it only after checking split and label coverage.')

## Evaluate by participant, not by treating windows as independent people

Average paired seed differences within each held-out participant, then bootstrap participants. These intervals are conditional on the trained models and fixed training/calibration split; they do not quantify uncertainty across all possible training cohorts. Six held-out participants still give limited precision.

Report overall CKA, per-participant CKA, within-activity CKA, Procrustes error, covariance rank and fixed-ridge activity macro-F1/balanced accuracy. Activity probe metrics average over classes present in each test participant; missing training classes cause undefined scores and are not silently filled. Small conditional groups remain undefined. No nearest-neighbor retrieval claim is made from overlapping windows.

In [ ]:
if bundle is not None:
    subjects = bundle.subjects
    groups = ['architecture', 'width', 'regime', 'step']
    display(paired_contrasts(subjects, metric='cka', group_by=groups, unit='subject'))
    display(paired_contrasts(subjects, metric='within_activity_cka', group_by=groups, unit='subject'))
    display(subjects[['subject', 'repeat', 'condition', 'regime', 'step', 'n_windows',
                      'cka', 'within_activity_cka', 'n_valid_activity_groups',
                      'balanced_accuracy_a', 'balanced_accuracy_b', 'macro_f1_a', 'macro_f1_b']])
    summary = subjects[subjects.step == FIT.steps].groupby(['regime', 'subject', 'condition']).cka.mean().unstack('condition')
    summary.plot.bar(figsize=(11, 4), ylabel='Mean CKA across seeds', title='Held-out participant comparisons')
    plt.tight_layout(); plt.show()
else:
    print('No real results to analyze; no claim about real-data performance is implied.')

## Optional export

Preparing ZIP bytes is memory-only. No file or browser download is made by default. If enabled before training, explicit persistence stores checkpoints and result archives only in the directory you choose. Do not select a laptop directory unless you intentionally want files there.

The optional link below downloads only after a manual click. The dataset itself is not included in the result archive; its source hash, preprocessing configuration, splits and normalization are recorded. Remember the dataset attribution when sharing results.

In [ ]:
SHOW_DOWNLOAD_LINK = False
EXPORT_BYTES = bundle_zip(bundle) if bundle is not None else None
if EXPORT_BYTES is not None:
    print(f'{len(EXPORT_BYTES):,} result-export bytes in RAM. Saving enabled: {SAVE_RESULTS}')
    if SHOW_DOWNLOAD_LINK:
        import base64
        from IPython.display import HTML
        encoded = base64.b64encode(EXPORT_BYTES).decode()
        link = '<a download="real_sensor_validation.zip" href="data:application/zip;base64,' + encoded + '">Click to download</a>'
        display(HTML(link))
else:
    print('Nothing exported or saved.')